# Is Airbnb Listing Price Associated with Proximity to Subway Stations in NYC?

**DATA 227 — Data Visualization Project | Winter 2026**  
**Authors:** Cristian Garcia, Avital Mintz, Vedant Dangayach

---

## Research Question

New York City's subway system is the lifeblood of the city — connecting millions of residents and visitors to jobs, entertainment, and neighborhoods. For short-term rental guests, proximity to public transit can make or break a stay. But does the market actually reflect this? **Is the nightly price of an Airbnb listing associated with the number of nearby subway stations?**

This notebook merges two datasets — NYC Airbnb listings and MTA subway station locations — to investigate whether transit accessibility is priced into the short-term rental market. We control for key confounders including borough, room type, and listing size, following a causal framework (DAG) to structure our analysis.

## 1. Setup & Data Loading

We begin by importing the libraries needed for each stage of the analysis:
- **pandas / numpy** for data wrangling and numeric computation
- **matplotlib / seaborn** for static, publication-quality plots (used in the notebook)
- **plotly** for interactive maps and charts (used in the Streamlit app)
- **geopandas** for handling GeoJSON borough boundaries
- **scipy.spatial.cKDTree** for efficient nearest-neighbor spatial queries (Haversine distance)
- **statsmodels** for OLS regression with full summary statistics

Two datasets are loaded: NYC Airbnb listings (Inside Airbnb / Kaggle, 2024 snapshot) and MTA subway station locations (NYC Open Data).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import geopandas as gpd
import json
from scipy.spatial import cKDTree
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# Plot styling
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

print('All packages loaded successfully.')

In [ ]:
# Load datasets
airbnb = pd.read_csv('data/new_york_listings_2024.csv')
subway = pd.read_csv('data/MTA_Subway_Stations.csv')

print(f'Airbnb listings: {airbnb.shape[0]:,} rows, {airbnb.shape[1]} columns')
print(f'Subway stations: {subway.shape[0]:,} rows, {subway.shape[1]} columns')

## 2. Data Exploration

Before merging, let's understand each dataset individually — its structure, distributions, and any cleaning needed.

### 2.1 Airbnb Listings

The Airbnb dataset contains approximately 20,000 listings scraped from Inside Airbnb's New York City page. Each row represents a single listing with attributes including nightly price, geographic coordinates, borough, neighborhood, room type, bedroom/bed count, guest rating, and review activity. We inspect the structure and summary statistics below to identify data quality issues and establish baseline distributions before merging with the subway data.

In [ ]:
airbnb.head(3)

In [ ]:
airbnb.info()

In [ ]:
# Key summary statistics for numeric columns
airbnb[['price', 'minimum_nights', 'number_of_reviews', 'reviews_per_month',
        'availability_365', 'rating', 'bedrooms', 'beds']].describe().round(2)

In [ ]:
# Borough breakdown
print('Listings by borough:')
print(airbnb['neighbourhood_group'].value_counts())
print()
print('Room type breakdown:')
print(airbnb['room_type'].value_counts())

### 2.2 MTA Subway Stations

The MTA dataset lists every subway station in the system with its geographic coordinates (GTFS Latitude/Longitude), stop name, and borough abbreviation. Because a single physical station can serve multiple lines, some locations appear more than once — we will deduplicate by coordinates in the cleaning step. Borough abbreviations (M, Bk, Q, Bx, SI) are mapped to full names for consistency with the Airbnb data.

In [ ]:
subway.head(3)

In [ ]:
# Borough mapping: MTA uses abbreviations
borough_map = {'M': 'Manhattan', 'Bk': 'Brooklyn', 'Q': 'Queens', 'Bx': 'Bronx', 'SI': 'Staten Island'}
subway['Borough_Full'] = subway['Borough'].map(borough_map)

print('Subway stations by borough:')
print(subway['Borough_Full'].value_counts())

## 3. Data Cleaning

Reliable analysis requires clean inputs. Our cleaning strategy addresses four issues:

1. **Missing essentials:** Rows without price or coordinates are dropped — they cannot contribute to spatial analysis.
2. **Zero-price listings:** These are likely placeholder or erroneous entries and are removed.
3. **Extreme outliers:** Prices are capped at the 99th percentile (`price_capped`) to prevent a handful of luxury listings from distorting means and regression estimates. The original `price` column is preserved for reference.
4. **Type coercion:** Bedroom counts (including "Studio" → 0), bed counts, and ratings are converted to numeric types. Missing bedroom/bed values are imputed with the median for the listing's room type — a conservative choice that avoids introducing bias toward any borough.

For the subway data, we deduplicate by latitude/longitude to get unique physical station locations.

In [ ]:
# Check missing values in key columns
cols_of_interest = ['price', 'latitude', 'longitude', 'neighbourhood_group',
                    'room_type', 'bedrooms', 'beds', 'rating']
print('Missing values in key Airbnb columns:')
print(airbnb[cols_of_interest].isnull().sum())

In [ ]:
# Clean the Airbnb data
df = airbnb.copy()

# Drop rows without price or coordinates (essential for our analysis)
df = df.dropna(subset=['price', 'latitude', 'longitude'])

# Remove listings with price = 0 (likely errors)
df = df[df['price'] > 0]

# Cap extreme price outliers at the 99th percentile for visualization
# (we keep the originals for reference but create a capped version)
p99 = df['price'].quantile(0.99)
print(f'99th percentile price: ${p99:,.0f}')
print(f'Max price before capping: ${df["price"].max():,.0f}')
df['price_capped'] = df['price'].clip(upper=p99)

# Clean the rating column (strip whitespace if present)
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')

# Clean bedrooms: "Studio" → 0, convert to numeric
df['bedrooms'] = df['bedrooms'].replace('Studio', 0)
df['bedrooms'] = pd.to_numeric(df['bedrooms'], errors='coerce')

# Clean baths: "Not specified" → NaN, convert to numeric
df['baths'] = df['baths'].replace('Not specified', np.nan)
df['baths'] = pd.to_numeric(df['baths'], errors='coerce')

# Fill missing bedrooms/beds with median by room type
for col in ['bedrooms', 'beds']:
    df[col] = df.groupby('room_type')[col].transform(lambda x: x.fillna(x.median()))

print(f'\nCleaned dataset: {df.shape[0]:,} listings')

In [ ]:
# Clean subway data — keep unique station locations
# Some stations have multiple entries for different lines; we want unique physical locations
subway_clean = subway.drop_duplicates(subset=['GTFS Latitude', 'GTFS Longitude'])
print(f'Unique subway station locations: {subway_clean.shape[0]}')

## 4. Spatial Merge: Counting Nearby Subway Stations

The core of our analysis is linking Airbnb listings to nearby subway stations. We use the **Haversine formula** to calculate the great-circle distance between each listing and every subway station, then count how many stations fall within a fixed radius.

We use a **0.5-mile radius** (~800 meters) as our primary measure — roughly a 10-minute walk — which is the standard threshold for "transit-accessible" in urban planning. We also compute counts at 0.25-mile and 1-mile radii for robustness checks.

In [ ]:
def haversine_vectorized(lat1, lon1, lat2, lon2):
    """
    Compute the Haversine distance (in miles) between two sets of coordinates.
    Uses vectorized numpy operations for efficiency.
    """
    R = 3958.8  # Earth's radius in miles
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    return 2 * R * np.arcsin(np.sqrt(a))


# Use scipy's cKDTree for efficient nearest-neighbor search
# Convert lat/lon to radians for the tree
subway_coords = np.radians(subway_clean[['GTFS Latitude', 'GTFS Longitude']].values)
listing_coords = np.radians(df[['latitude', 'longitude']].values)

# Build the tree from subway station coordinates
tree = cKDTree(subway_coords)

# Earth's radius in miles
R = 3958.8

# Query for each radius
for radius_miles, col_name in [(0.25, 'stations_025mi'),
                                (0.5, 'stations_05mi'),
                                (1.0, 'stations_1mi')]:
    # Convert miles to radians for the ball tree query
    radius_rad = radius_miles / R
    counts = tree.query_ball_point(listing_coords, r=radius_rad)
    df[col_name] = [len(c) for c in counts]

# Also find distance to the nearest station
dist_rad, idx = tree.query(listing_coords, k=1)
df['nearest_station_miles'] = dist_rad * R
df['nearest_station_name'] = subway_clean.iloc[idx]['Stop Name'].values

print('Subway proximity features added.')
print()
print(df[['stations_025mi', 'stations_05mi', 'stations_1mi', 'nearest_station_miles']].describe().round(2))

## 5. Exploratory Visualizations

With the merged dataset in hand, we now explore the data visually — building from simple distributions toward the central question. Each visualization is chosen to reveal a specific facet of the data:

- **Box and violin plots** expose distributional shape, median, and outliers across categories — richer than bar charts of means alone.
- **Stacked bar charts** show both absolute volume and proportional composition simultaneously.
- **Scatter mapbox plots** preserve geographic relationships that tabular summaries would lose.
- **Choropleths** leverage spatial familiarity with NYC's borough boundaries to make price gradients immediately intuitive.

We start with univariate price distributions, move to spatial maps, and then examine the price–subway relationship directly.

### 5.1 Price Distribution by Borough

Before examining subway proximity, we need to understand the baseline price landscape. Borough is the strongest geographic confounder — Manhattan listings command premium prices regardless of subway access.

In [ ]:
# Price distribution by borough — box plot
borough_order = df.groupby('neighbourhood_group')['price_capped'].median().sort_values(ascending=False).index

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Box plot
sns.boxplot(data=df, x='neighbourhood_group', y='price_capped', order=borough_order,
            palette='Set2', ax=axes[0], fliersize=1)
axes[0].set_title('Airbnb Nightly Price Distribution by Borough', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Borough')
axes[0].set_ylabel('Price ($ per night, capped at 99th pctile)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Violin plot for shape
sns.violinplot(data=df, x='neighbourhood_group', y='price_capped', order=borough_order,
               palette='Set2', ax=axes[1], inner='quartile', cut=0)
axes[1].set_title('Price Distribution Shape by Borough', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Borough')
axes[1].set_ylabel('Price ($ per night, capped at 99th pctile)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig('viz_price_by_borough.png', bbox_inches='tight', dpi=150)
plt.show()
print('Manhattan has the highest median price, followed by Brooklyn. Staten Island is the most affordable.')

### 5.2 Room Type Composition by Borough

Room type is another key confounder. Entire apartments cost more than private rooms, and boroughs differ in their room type mix.

In [ ]:
# Stacked bar chart: room type composition by borough
room_counts = df.groupby(['neighbourhood_group', 'room_type']).size().unstack(fill_value=0)
room_pcts = room_counts.div(room_counts.sum(axis=1), axis=0) * 100
room_pcts = room_pcts.loc[borough_order]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Absolute counts
room_counts.loc[borough_order].plot(kind='bar', stacked=True, ax=axes[0],
                                     colormap='Set2', edgecolor='white')
axes[0].set_title('Number of Listings by Borough & Room Type', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Borough')
axes[0].set_ylabel('Number of Listings')
axes[0].legend(title='Room Type', bbox_to_anchor=(1.0, 1.0))
axes[0].tick_params(axis='x', rotation=0)

# Percentage
room_pcts.plot(kind='bar', stacked=True, ax=axes[1],
               colormap='Set2', edgecolor='white')
axes[1].set_title('Room Type Share by Borough (%)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Borough')
axes[1].set_ylabel('Percentage of Listings')
axes[1].legend(title='Room Type', bbox_to_anchor=(1.0, 1.0))
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('viz_roomtype_by_borough.png', bbox_inches='tight', dpi=150)
plt.show()

### 5.3 Subway Station Density by Borough

How are subway stations distributed across the city? Manhattan is famously the most transit-rich borough, while Staten Island has only the SIR line.

In [ ]:
# Subway stations by borough
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Station count
station_counts = subway_clean['Borough_Full'].value_counts().reindex(
    ['Manhattan', 'Brooklyn', 'Queens', 'Bronx', 'Staten Island'])
colors = sns.color_palette('Set2', n_colors=5)
station_counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_title('Number of Subway Station Locations by Borough', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Borough')
axes[0].set_ylabel('Number of Unique Station Locations')
axes[0].tick_params(axis='x', rotation=0)

# Average nearby stations per listing by borough
avg_stations = df.groupby('neighbourhood_group')['stations_05mi'].mean().reindex(
    ['Manhattan', 'Brooklyn', 'Queens', 'Bronx', 'Staten Island'])
avg_stations.plot(kind='bar', ax=axes[1], color=colors, edgecolor='black', linewidth=0.5)
axes[1].set_title('Avg. Subway Stations within 0.5 mi of Listings', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Borough')
axes[1].set_ylabel('Mean Stations within 0.5 miles')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('viz_subway_density.png', bbox_inches='tight', dpi=150)
plt.show()

### 5.4 Interactive Map: Airbnb Listings and Subway Stations

A geographic scatter plot is essential for our analysis because the central question is inherently spatial: does *proximity* to transit infrastructure predict price? A table of correlations cannot convey what a map can — namely, the visual overlap between expensive listings (red) and dense subway clusters (blue). We sample up to 5,000 listings for Plotly rendering performance and use the `carto-positron` basemap for a clean, low-contrast background that keeps data marks legible.

In [ ]:
# Sample listings for performance (plotly can struggle with 20k+ points)
sample = df.sample(n=min(5000, len(df)), random_state=42)

fig = px.scatter_mapbox(
    sample,
    lat='latitude',
    lon='longitude',
    color='price_capped',
    color_continuous_scale='YlOrRd',
    size_max=6,
    opacity=0.5,
    hover_name='name',
    hover_data={'price': ':$.0f', 'neighbourhood_group': True, 'room_type': True,
                'stations_05mi': True, 'latitude': False, 'longitude': False},
    labels={'price_capped': 'Price ($)', 'stations_05mi': 'Stations (0.5mi)'},
    title='NYC Airbnb Listings Colored by Price'
)

# Add subway stations as a separate trace
fig.add_trace(go.Scattermapbox(
    lat=subway_clean['GTFS Latitude'],
    lon=subway_clean['GTFS Longitude'],
    mode='markers',
    marker=dict(size=6, color='blue', opacity=0.7),
    name='Subway Stations',
    text=subway_clean['Stop Name'],
    hoverinfo='text'
))

fig.update_layout(
    mapbox_style='carto-positron',
    mapbox_center={'lat': 40.7128, 'lon': -74.0060},
    mapbox_zoom=10,
    height=700,
    margin=dict(l=0, r=0, t=40, b=0)
)
fig.show()

### 5.5 Choropleth Map: Median Price by Borough

A choropleth encodes a numeric variable (here, median price) as fill color across predefined geographic regions. Borough boundaries are familiar to most NYC-aware viewers, making the price gradient immediately interpretable without requiring a coordinate lookup. We use median rather than mean price to reduce the influence of extreme luxury outliers that are concentrated in Manhattan.

In [ ]:
# Load borough GeoJSON
with open('data/nyc-borough.geojson', 'r') as f:
    borough_geo = json.load(f)

# Compute median price by borough
borough_prices = df.groupby('neighbourhood_group').agg(
    median_price=('price', 'median'),
    mean_price=('price', 'mean'),
    listing_count=('price', 'count'),
    median_stations=('stations_05mi', 'median')
).reset_index()

# Standardize borough names to match GeoJSON
# Check what names the GeoJSON uses
geojson_names = [f['properties'].get('name', f['properties'].get('boro_name', ''))
                 for f in borough_geo['features']]
print('GeoJSON borough names:', geojson_names)
print('Data borough names:', borough_prices['neighbourhood_group'].tolist())

In [ ]:
# Build choropleth
fig = px.choropleth_mapbox(
    borough_prices,
    geojson=borough_geo,
    locations='neighbourhood_group',
    featureidkey='properties.name',  # adjust if needed based on output above
    color='median_price',
    color_continuous_scale='YlOrRd',
    hover_data={'mean_price': ':$.0f', 'listing_count': ':,', 'median_stations': True},
    labels={'median_price': 'Median Price ($)', 'mean_price': 'Mean Price ($)',
            'listing_count': 'Listings', 'median_stations': 'Median Stations (0.5mi)'},
    title='Median Airbnb Nightly Price by Borough'
)
fig.update_layout(
    mapbox_style='carto-positron',
    mapbox_center={'lat': 40.7128, 'lon': -74.0060},
    mapbox_zoom=9.5,
    height=600,
    margin=dict(l=0, r=0, t=40, b=0)
)
fig.show()

## 6. Core Analysis: Price vs. Subway Proximity

We now turn to the central question: does the number of nearby subway stations predict Airbnb price? The exploratory visualizations above established that Manhattan has both the highest prices and the densest subway coverage — creating a confound that must be carefully addressed. Below, we examine the price–station relationship at multiple levels of aggregation: raw scatter plots, binned summaries, borough-stratified views, a correlation matrix, and distance-based analysis. Each layer peels back a confound and sharpens the picture.

### 6.1 Scatter Plot: Price vs. Nearby Stations

We begin with the simplest view: scatter plots of price against station count at three radii (0.25, 0.5, and 1 mile). The point cloud will be heavily over-plotted — which is itself informative, as it reveals the density of the data. A red mean-price line overlaid on each panel shows the central tendency at each station count, cutting through the noise to expose the trend direction.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, (col, label) in zip(axes, [
    ('stations_025mi', '0.25-mile radius'),
    ('stations_05mi', '0.5-mile radius'),
    ('stations_1mi', '1-mile radius')
]):
    # Jitter for visibility
    jitter = np.random.normal(0, 0.15, size=len(df))
    ax.scatter(df[col] + jitter, df['price_capped'], alpha=0.05, s=3, color='steelblue')

    # Add mean line
    means = df.groupby(col)['price_capped'].mean()
    ax.plot(means.index, means.values, color='red', linewidth=2.5, marker='o',
            markersize=6, label='Mean price', zorder=5)

    ax.set_title(f'Price vs. Stations ({label})', fontsize=13, fontweight='bold')
    ax.set_xlabel(f'Number of Stations within {label}')
    ax.set_ylabel('Price ($ per night)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax.legend()

plt.tight_layout()
plt.savefig('viz_price_vs_stations_scatter.png', bbox_inches='tight', dpi=150)
plt.show()

### 6.2 Binned Bar Chart: Average Price by Station Count

Scatter plots show the full data but can be noisy. Binning station counts into ordered groups (0, 1–2, 3–5, 6–10, 10+) and plotting the mean and median price per bin produces a cleaner summary. The sample size label on each bar (n=) signals where the estimates are reliable and where they rest on thin data — an important transparency measure, since the highest-station bins often contain far fewer listings.

In [ ]:
# Create station count bins
df['station_bin'] = pd.cut(df['stations_05mi'],
                           bins=[-1, 0, 2, 5, 10, 100],
                           labels=['0', '1-2', '3-5', '6-10', '10+'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Mean price by bin
bin_stats = df.groupby('station_bin', observed=True).agg(
    mean_price=('price_capped', 'mean'),
    median_price=('price_capped', 'median'),
    count=('price_capped', 'count')
).reset_index()

# Bar chart — mean price
bars = axes[0].bar(bin_stats['station_bin'], bin_stats['mean_price'],
                   color=sns.color_palette('YlOrRd', n_colors=len(bin_stats)),
                   edgecolor='black', linewidth=0.5)
axes[0].set_title('Mean Nightly Price by Nearby Subway Stations (0.5 mi)',
                  fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Stations within 0.5 miles')
axes[0].set_ylabel('Mean Price ($)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
# Add count labels on bars
for bar, count in zip(bars, bin_stats['count']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                f'n={count:,}', ha='center', va='bottom', fontsize=9)

# Bar chart — median price
bars2 = axes[1].bar(bin_stats['station_bin'], bin_stats['median_price'],
                    color=sns.color_palette('YlOrRd', n_colors=len(bin_stats)),
                    edgecolor='black', linewidth=0.5)
axes[1].set_title('Median Nightly Price by Nearby Subway Stations (0.5 mi)',
                  fontsize=13, fontweight='bold')
axes[1].set_xlabel('Number of Stations within 0.5 miles')
axes[1].set_ylabel('Median Price ($)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
for bar, count in zip(bars2, bin_stats['count']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                f'n={count:,}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('viz_price_by_station_bins.png', bbox_inches='tight', dpi=150)
plt.show()

### 6.3 Price vs. Stations — Controlling for Borough

Borough is the primary confounder: Manhattan has both the highest prices *and* the most subway stations. To isolate the effect of subway proximity, we need to look *within* each borough.

In [ ]:
# FacetGrid: Price vs stations by borough
g = sns.FacetGrid(df, col='neighbourhood_group', col_wrap=3, height=4.5, aspect=1.2,
                  col_order=['Manhattan', 'Brooklyn', 'Queens', 'Bronx', 'Staten Island'])

def plot_with_mean(data, **kwargs):
    jitter = np.random.normal(0, 0.15, size=len(data))
    plt.scatter(data['stations_05mi'] + jitter, data['price_capped'],
                alpha=0.05, s=3, color='steelblue')
    means = data.groupby('stations_05mi')['price_capped'].mean()
    plt.plot(means.index, means.values, color='red', linewidth=2, marker='o',
             markersize=5, zorder=5)

g.map_dataframe(plot_with_mean)
g.set_axis_labels('Stations within 0.5 mi', 'Price ($)')
g.set_titles('{col_name}', fontsize=13, fontweight='bold')
g.figure.suptitle('Price vs. Nearby Subway Stations by Borough', fontsize=15,
                   fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig('viz_price_vs_stations_by_borough.png', bbox_inches='tight', dpi=150)
plt.show()

### 6.4 Heatmap: Correlation Between Key Variables

Before building a regression model, we need to understand pairwise relationships across all numeric variables. A correlation heatmap using a lower-triangle mask (to avoid redundancy) and a diverging RdBu_r color scale (blue = positive, red = negative, white = zero) lets us quickly identify which variables move together and which are near-independent. This informs variable selection for the regression: highly correlated predictors (e.g., stations at 0.5 mi and 1 mi) could introduce multicollinearity.

In [ ]:
# Correlation heatmap of key numeric variables
corr_cols = ['price_capped', 'stations_025mi', 'stations_05mi', 'stations_1mi',
             'nearest_station_miles', 'bedrooms', 'beds', 'rating',
             'number_of_reviews', 'reviews_per_month', 'availability_365',
             'minimum_nights']
corr_labels = ['Price', 'Stations (0.25mi)', 'Stations (0.5mi)', 'Stations (1mi)',
               'Nearest Stn (mi)', 'Bedrooms', 'Beds', 'Rating',
               'Reviews', 'Reviews/mo', 'Availability', 'Min Nights']

corr_matrix = df[corr_cols].corr()
corr_matrix.index = corr_labels
corr_matrix.columns = corr_labels

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8, 'label': 'Pearson Correlation'})
ax.set_title('Correlation Heatmap of Key Variables', fontsize=15, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('viz_correlation_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()

### 6.5 Distance to Nearest Station vs. Price

Station count measures density of access; distance to the *nearest* station measures isolation. If transit proximity is genuinely valued by guests, we expect prices to decline as distance to the closest station increases. We bin distance into five intuitive categories (<0.1 mi, 0.1–0.25, 0.25–0.5, 0.5–1, >1 mi) and use a box plot alongside a count chart. The count chart matters: if a distance bin contains very few listings, its price distribution is unreliable.

In [ ]:
# Bin by distance to nearest station
df['dist_bin'] = pd.cut(df['nearest_station_miles'],
                        bins=[0, 0.1, 0.25, 0.5, 1.0, df['nearest_station_miles'].max()],
                        labels=['<0.1 mi', '0.1-0.25 mi', '0.25-0.5 mi', '0.5-1 mi', '>1 mi'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Box plot by distance bin
sns.boxplot(data=df, x='dist_bin', y='price_capped', palette='coolwarm_r',
            ax=axes[0], fliersize=1)
axes[0].set_title('Price by Distance to Nearest Subway Station', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Distance to Nearest Station')
axes[0].set_ylabel('Price ($ per night)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Count by distance bin
dist_counts = df['dist_bin'].value_counts().reindex(
    ['<0.1 mi', '0.1-0.25 mi', '0.25-0.5 mi', '0.5-1 mi', '>1 mi'])
dist_counts.plot(kind='bar', ax=axes[1], color=sns.color_palette('coolwarm_r', n_colors=5),
                 edgecolor='black', linewidth=0.5)
axes[1].set_title('Number of Listings by Distance to Nearest Station',
                  fontsize=13, fontweight='bold')
axes[1].set_xlabel('Distance to Nearest Station')
axes[1].set_ylabel('Number of Listings')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('viz_price_by_distance.png', bbox_inches='tight', dpi=150)
plt.show()

### 6.6 Neighborhood-Level Bubble Chart

Borough-level analysis groups very different neighborhoods together. This interactive bubble chart zooms in to individual neighborhoods, encoding three variables simultaneously: average station density (x-axis), median price (y-axis), and listing count (bubble size). Color marks borough membership. This multidimensional encoding follows the principle that scatter plots are effective for revealing clusters, outliers, and subgroup patterns — here, we can spot neighborhoods that break the expected transit-price pattern.

In [ ]:
# Neighborhood-level stats
nbhd_stats = df.groupby('neighbourhood').agg(
    median_price=('price', 'median'),
    mean_stations=('stations_05mi', 'mean'),
    count=('price', 'count'),
    borough=('neighbourhood_group', 'first')
).reset_index()

# Only neighborhoods with enough listings for stable estimates
nbhd_stats = nbhd_stats[nbhd_stats['count'] >= 20]

fig = px.scatter(
    nbhd_stats,
    x='mean_stations',
    y='median_price',
    size='count',
    color='borough',
    hover_name='neighbourhood',
    labels={'mean_stations': 'Avg Stations within 0.5 mi',
            'median_price': 'Median Price ($)',
            'count': 'Listings',
            'borough': 'Borough'},
    title='Neighborhood-Level: Median Price vs. Subway Accessibility',
    size_max=30,
    opacity=0.7
)
fig.update_layout(height=600)
fig.show()

## 7. Statistical Analysis: Regression

The visualizations above suggest a positive but confounded relationship between subway proximity and price. To quantify the association rigorously — and to isolate it from borough, room type, and listing size effects — we estimate three OLS regression models of increasing complexity.

Our specification follows a DAG (Directed Acyclic Graph) causal framework:
- **Outcome (Y):** Nightly price (capped at 99th percentile)
- **Treatment (X):** Number of subway stations within 0.5 miles
- **Confounders:** Borough (strongest geographic confounder), room type, bedrooms, beds

By progressively adding controls, we can observe how the subway coefficient changes — revealing how much of the raw association is genuine vs. driven by confounding variables.

In [ ]:
# Prepare data for regression
reg_df = df[['price_capped', 'stations_05mi', 'neighbourhood_group',
             'room_type', 'bedrooms', 'beds']].dropna().copy()

# Ensure numeric types
reg_df['bedrooms'] = pd.to_numeric(reg_df['bedrooms'], errors='coerce')
reg_df['beds'] = pd.to_numeric(reg_df['beds'], errors='coerce')
reg_df = reg_df.dropna()

# Create dummy variables for categorical controls
reg_df = pd.get_dummies(reg_df, columns=['neighbourhood_group', 'room_type'], drop_first=True)

# Ensure all columns are float
for col in reg_df.columns:
    reg_df[col] = reg_df[col].astype(float)

# Model 1: Simple bivariate
X1 = sm.add_constant(reg_df[['stations_05mi']])
y = reg_df['price_capped']
model1 = sm.OLS(y, X1).fit()

# Model 2: With borough controls
borough_cols = [c for c in reg_df.columns if c.startswith('neighbourhood_group_')]
X2 = sm.add_constant(reg_df[['stations_05mi'] + borough_cols])
model2 = sm.OLS(y, X2).fit()

# Model 3: Full model with all controls
room_cols = [c for c in reg_df.columns if c.startswith('room_type_')]
X3 = sm.add_constant(reg_df[['stations_05mi', 'bedrooms', 'beds'] + borough_cols + room_cols])
model3 = sm.OLS(y, X3).fit()

print('='*70)
print('MODEL 1: Price ~ Stations (bivariate)')
print('='*70)
print(f'Coefficient on stations_05mi: {model1.params["stations_05mi"]:.2f}')
print(f'  (p-value: {model1.pvalues["stations_05mi"]:.4f})')
print(f'R-squared: {model1.rsquared:.4f}')
print()
print('='*70)
print('MODEL 2: Price ~ Stations + Borough')
print('='*70)
print(f'Coefficient on stations_05mi: {model2.params["stations_05mi"]:.2f}')
print(f'  (p-value: {model2.pvalues["stations_05mi"]:.4f})')
print(f'R-squared: {model2.rsquared:.4f}')
print()
print('='*70)
print('MODEL 3: Price ~ Stations + Borough + Room Type + Bedrooms + Beds')
print('='*70)
print(f'Coefficient on stations_05mi: {model3.params["stations_05mi"]:.2f}')
print(f'  (p-value: {model3.pvalues["stations_05mi"]:.4f})')
print(f'R-squared: {model3.rsquared:.4f}')

In [ ]:
# Full regression table for the final model
print(model3.summary())

In [ ]:
# Visualize the regression coefficients across models
coef_data = pd.DataFrame({
    'Model': ['Bivariate', 'Borough Controls', 'Full Controls'],
    'Coefficient': [model1.params['stations_05mi'],
                    model2.params['stations_05mi'],
                    model3.params['stations_05mi']],
    'CI_low': [model1.conf_int().loc['stations_05mi', 0],
               model2.conf_int().loc['stations_05mi', 0],
               model3.conf_int().loc['stations_05mi', 0]],
    'CI_high': [model1.conf_int().loc['stations_05mi', 1],
                model2.conf_int().loc['stations_05mi', 1],
                model3.conf_int().loc['stations_05mi', 1]]
})

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2196F3', '#FF9800', '#4CAF50']
ax.barh(coef_data['Model'], coef_data['Coefficient'], color=colors,
        edgecolor='black', linewidth=0.5, height=0.5)
ax.errorbar(coef_data['Coefficient'], coef_data['Model'],
            xerr=[coef_data['Coefficient'] - coef_data['CI_low'],
                  coef_data['CI_high'] - coef_data['Coefficient']],
            fmt='none', color='black', capsize=5)
ax.axvline(x=0, color='black', linestyle='--', linewidth=0.8)
ax.set_xlabel('Coefficient on "Stations within 0.5 mi" ($ per station)', fontsize=12)
ax.set_title('Effect of Subway Proximity on Airbnb Price\nAcross Model Specifications',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('viz_regression_coefficients.png', bbox_inches='tight', dpi=150)
plt.show()

print('\nInterpretation: Each additional subway station within 0.5 miles is associated with')
print(f'a ${model3.params["stations_05mi"]:.2f} change in nightly price,'
      f' controlling for borough, room type, and listing size.')

## 8. Additional Visualizations

The core analysis above focused on the subway-price relationship. These supplementary visualizations round out the picture by examining the overall price distribution, the room-type premium, and borough-by-station interactions in a 2D heatmap. They also provide additional chart types (histograms, standalone box plots, annotated heatmaps) that demonstrate breadth of visualization technique.

### 8.1 Price Distribution (Overall)

Understanding the overall shape of the price distribution is a prerequisite for choosing appropriate summary statistics and regression models. A right-skewed distribution (common for prices) means the mean is pulled upward by outliers, making the median a more robust central measure. The log-transformed histogram on the right checks whether the distribution is approximately log-normal — which would support the use of linear regression on capped prices as a reasonable approximation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Histogram of price
axes[0].hist(df['price_capped'], bins=80, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(df['price_capped'].median(), color='red', linestyle='--', linewidth=2,
                label=f'Median: ${df["price_capped"].median():,.0f}')
axes[0].axvline(df['price_capped'].mean(), color='orange', linestyle='--', linewidth=2,
                label=f'Mean: ${df["price_capped"].mean():,.0f}')
axes[0].set_title('Distribution of Airbnb Nightly Prices (NYC)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Price ($ per night)')
axes[0].set_ylabel('Frequency')
axes[0].legend(fontsize=11)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Log-transformed
axes[1].hist(np.log10(df['price_capped']), bins=80, color='darkorange', edgecolor='white', alpha=0.8)
axes[1].set_title('Log₁₀ Price Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Log₁₀(Price)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('viz_price_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

### 8.2 Price by Room Type

Room type is one of the strongest predictors of price and a key control variable in our regression. This box plot makes the price hierarchy explicit — entire homes/apartments command a clear premium over private rooms, which in turn exceed shared rooms and hotel rooms. The spread within each category also matters: entire homes show the widest interquartile range, reflecting the diversity from studio apartments to multi-bedroom units.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
room_order = df.groupby('room_type')['price_capped'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='room_type', y='price_capped', order=room_order,
            palette='pastel', ax=ax, fliersize=1)
ax.set_title('Price Distribution by Room Type', fontsize=14, fontweight='bold')
ax.set_xlabel('Room Type')
ax.set_ylabel('Price ($ per night)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig('viz_price_by_roomtype.png', bbox_inches='tight', dpi=150)
plt.show()

### 8.3 Heatmap: Avg Price by Borough x Station Bin

This 2D heatmap reveals how price varies across both borough and subway accessibility simultaneously — the core interaction in our analysis.

In [ ]:
# Pivot: average price by borough and station bin
pivot = df.pivot_table(values='price_capped', index='neighbourhood_group',
                       columns='station_bin', aggfunc='median', observed=True)
pivot = pivot.reindex(['Manhattan', 'Brooklyn', 'Queens', 'Bronx', 'Staten Island'])

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd', linewidths=1,
            ax=ax, cbar_kws={'label': 'Median Price ($)'})
ax.set_title('Median Price: Borough x Subway Station Accessibility',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Stations within 0.5 miles')
ax.set_ylabel('Borough')
plt.tight_layout()
plt.savefig('viz_heatmap_borough_stations.png', bbox_inches='tight', dpi=150)
plt.show()

### 8.4 Interactive Map: Station Density Heatmap

A density mapbox uses kernel density estimation to transform discrete listing locations into a continuous surface, revealing spatial concentration patterns that individual scatter points cannot. The Viridis color scale is chosen for its perceptual uniformity and accessibility to colorblind viewers. High-intensity zones correspond to areas where many listings have high station counts — effectively a map of where transit accessibility and Airbnb supply overlap most.

In [ ]:
# Plotly density map of listings colored by station count
sample2 = df.sample(n=min(8000, len(df)), random_state=42)

fig = px.density_mapbox(
    sample2,
    lat='latitude',
    lon='longitude',
    z='stations_05mi',
    radius=8,
    mapbox_style='carto-positron',
    center={'lat': 40.7128, 'lon': -74.0060},
    zoom=10,
    title='Subway Accessibility Heatmap (Stations within 0.5 mi per listing)',
    color_continuous_scale='Viridis',
    labels={'stations_05mi': 'Stations'}
)
fig.update_layout(height=600, margin=dict(l=0, r=0, t=40, b=0))
fig.show()

## 9. Summary of Findings

### Key Takeaways

1. **Borough dominates pricing:** Manhattan listings are significantly more expensive than other boroughs, and Manhattan also has the densest subway coverage. This creates a strong confound that must be controlled for.

2. **Subway proximity is correlated with price in raw data:** Listings near more subway stations tend to have higher prices. However, much of this association is driven by the borough-level confound.

3. **Within-borough effect:** After controlling for borough, room type, and listing size, the association between subway proximity and price becomes clearer. The regression results show the magnitude and significance of this effect.

4. **Room type and size matter:** Entire homes/apartments command higher prices, and larger listings (more bedrooms/beds) are more expensive. These are important controls.

5. **Spatial patterns are rich:** The interactive maps reveal neighborhood-level variation that summary statistics alone cannot capture.

### Limitations

- **Observational data:** We cannot claim causation. Subway stations may be correlated with other neighborhood amenities (restaurants, attractions) that also drive price.
- **Missing variables:** We lack data on listing quality (photos, amenities), seasonality, and specific listing features that affect pricing.
- **Cross-sectional snapshot:** This is a single point-in-time analysis; prices and station access may change over time.
- **Distance metric:** Walking distance differs from straight-line distance; actual transit accessibility depends on service frequency and transfers.

### Future Directions

- Incorporate time-series data to study seasonal effects
- Add neighborhood-level controls (median income, crime rates, walkability scores)
- Use instrumental variables or natural experiments (new station openings) for causal inference
- Compare with other transit modes (bus, bike stations)

## 10. Export Merged Dataset

We save the fully cleaned and feature-engineered dataset for use in the Streamlit interactive dashboard (`app.py`). This ensures the app loads pre-computed subway proximity features rather than recalculating them at runtime, improving performance and guaranteeing reproducibility — the app always uses exactly the same data as this notebook.

In [ ]:
# Save the merged and cleaned dataset
df.to_csv('data/airbnb_with_subway_features.csv', index=False)
print(f'Saved merged dataset: {df.shape[0]:,} rows, {df.shape[1]} columns')
print(f'File: data/airbnb_with_subway_features.csv')